In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from ann_model_massspecgym import AnnRetrieval
from distance_functions import cosine_similarity, euclidean_distance
from chemembed_transforms import ChemEmbedSpecTransform, molformer_factory
from massspecgym.data.datasets import MassSpecDataset
from massspecgym.data.data_module import MassSpecDataModule
from massspecgym.data.transforms import InMemCachedMolTransform
# rdkit.Chem.Scaffolds.MurckoScaffold

from config import DATA_DIR, WORKING_DIR

# 1) MoLFormer Embedding

In [7]:
# MoLFormer (768 dims)
mol_transform_factory = molformer_factory
embedding_name = "molformer"
mol_embedding_dim = 768

selected_molecular_embedding = InMemCachedMolTransform(
    mol_transform_factory=mol_transform_factory,
    verbose=True
)

Loading weights:   0%|          | 0/207 [00:00<?, ?it/s]

[transformers] MolformerModel LOAD REPORT from: ibm/MoLFormer-XL-both-10pct
Key                                | Status     |  | 
-----------------------------------+------------+--+-
lm_head.transform.LayerNorm.weight | UNEXPECTED |  | 
lm_head.transform.dense.weight     | UNEXPECTED |  | 
lm_head.decoder.weight             | UNEXPECTED |  | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED |  | 
lm_head.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initializing InMemCachedMolTransform for MoLFormerMolTransform. Cache file not provided or missing; molecules will be transformed on the fly. Call `build_cache(smiles_list)` to create it.


## Calculate embeddings

In [ ]:
dataset_path = Path(DATA_DIR, WORKING_DIR, "MSG_w_chemont.tsv")
transformer = mol_transform_factory()

df = pd.read_csv(dataset_path, sep="\t")
df["molformer_embedding"] = df["smiles"].apply(transformer.from_smiles)
df.info()

Loading weights:   0%|          | 0/207 [00:00<?, ?it/s]

[transformers] MolformerModel LOAD REPORT from: ibm/MoLFormer-XL-both-10pct
Key                                | Status     |  | 
-----------------------------------+------------+--+-
lm_head.transform.dense.weight     | UNEXPECTED |  | 
lm_head.transform.LayerNorm.weight | UNEXPECTED |  | 
lm_head.transform.dense.bias       | UNEXPECTED |  | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14066 entries, 0 to 14065
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   identifier            14066 non-null  object 
 1   mzs                   14066 non-null  object 
 2   intensities           14066 non-null  object 
 3   smiles                14066 non-null  object 
 4   inchikey              14066 non-null  object 
 5   formula               14066 non-null  object 
 6   precursor_formula     14066 non-null  object 
 7   parent_mass           14066 non-null  float64
 8   precursor_mz          14066 non-null  float64
 9   adduct                14066 non-null  object 
 10  instrument_type       13816 non-null  object 
 11  collision_energy      9957 non-null   float64
 12  fold                  14066 non-null  object 
 13  simulation_challenge  14066 non-null  bool   
 14  chemont_tree          14056 non-null  object 
 15  molformer_embedding

In [ ]:
new_path = Path(str(dataset_path.with_suffix('')) + "_plus_embedding.tsv")
df.to_csv(new_path, sep="\t", index=False)

# 2) Load Dataset

In [ ]:
working_dataset = MassSpecDataset(
    pth=dataset_path,
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
)

data_module_test = MassSpecDataModule(
    dataset=working_dataset,
    batch_size=1,
)

data_module_test.setup("test")

# 3) Model Evaluation

## Load model

In [9]:
model_name = "ANN_trained_molformer.ckpt"
model = AnnRetrieval.load_from_checkpoint(
    f"models/{model_name}",
    mol_embedding_dim=mol_embedding_dim
)

with torch.no_grad():
    print(model.eval())

c:\Users\danay\Documents\Code\MassSpecGym\.milab\Lib\site-packages\lightning_fabric\utilities\cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


AnnRetrieval(
  (ann): ANN_Class(
    (fc1): Linear(in_features=70000, out_features=2048, bias=True)
    (fc2): Linear(in_features=2048, out_features=1024, bias=True)
    (fc3): Linear(in_features=1024, out_features=768, bias=True)
    (dropout): Dropout(p=0.25, inplace=False)
  )
)


## Predict MoLFormer Embeddings

In [ ]:
model_predictions = []
with torch.no_grad():
    for entry in tqdm(data_module_test.test_dataloader()):
        input = entry["spec"]
        if (torch.isnan(input).any()):
            input = torch.from_numpy(np.array([0] * mol_embedding_dim).astype(np.float32))
        input = input.to("cpu") 
        outputs = model(input).cpu()
        model_predictions.append(outputs)

pred_df = pd.concat([df, pd.DataFrame({"molformer_embedding_pred" : model_predictions}).reset_index(drop=True)], axis=1)
pred_df

In [28]:
new_path = Path(str(dataset_path.with_suffix('')) + "_plus_embedding_pred.tsv")
pred_df.to_csv(new_path, sep="\t", index=False)

## Compare with real embedding

In [25]:
ids = list(pred_df['identifier'])
pred_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14066 entries, 0 to 14065
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   identifier                14066 non-null  object 
 1   mzs                       14066 non-null  object 
 2   intensities               14066 non-null  object 
 3   smiles                    14066 non-null  object 
 4   inchikey                  14066 non-null  object 
 5   formula                   14066 non-null  object 
 6   precursor_formula         14066 non-null  object 
 7   parent_mass               14066 non-null  float64
 8   precursor_mz              14066 non-null  float64
 9   adduct                    14066 non-null  object 
 10  instrument_type           13816 non-null  object 
 11  collision_energy          9957 non-null   float64
 12  fold                      14066 non-null  object 
 13  simulation_challenge      14066 non-null  bool   
 14  chemon

In [26]:
results = list()

cosine = cosine_similarity(pred_df, "molformer_embedding", "molformer_embedding_pred")
results.append(pd.DataFrame({'identifier': ids, 'cosine_sim': cosine}))
euc = euclidean_distance(pred_df, "molformer_embedding", "molformer_embedding_pred")
results.append(pd.DataFrame({'identifier': ids, 'euclidean_dist': euc}))

print(len(cosine))
print(len(euc))
print(len(ids))

14066
14066
14066


In [27]:
measures = results[0]
for i in range(1, len(results)):
    measures = pd.merge(measures, results[i], on="identifier")

measures.info()
measures

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14066 entries, 0 to 14065
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   identifier      14066 non-null  object 
 1   cosine_sim      14066 non-null  float32
 2   euclidean_dist  14066 non-null  float32
dtypes: float32(2), object(1)
memory usage: 219.9+ KB


,identifier,cosine_sim,euclidean_dist
0,MassSpecGymID0000203,0.761149,10.036486
1,MassSpecGymID0000204,0.739842,10.529962
2,MassSpecGymID0000205,0.767589,9.961919
3,MassSpecGymID0000206,0.782684,9.603712
4,MassSpecGymID0000208,0.798874,9.288404
...,...,...,...
14061,MassSpecGymID0400854,0.653316,14.504745
14062,MassSpecGymID0401425,0.762149,11.733856
14063,MassSpecGymID0402033,0.757642,10.278358
14064,MassSpecGymID0402039,0.855012,8.529740


In [ ]:
measures.to_csv(Path(DATA_DIR, WORKING_DIR, "similarity_measures.tsv"), sep="\t", index=False)